In [4]:

from mpramnist.Rafi2024 import RafiDataset
from mpramnist.Rafi2024 import LitModel_Rafi

import mpramnist.transforms as t

from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

import torch
import torch.nn as nn
import torch.utils.data as data
from torchmetrics import PearsonCorrCoef

import lightning.pytorch as L

BATCH_SIZE = 1024
NUM_WORKERS = 8

length = 120
plasmid = RafiDataset.PLASMID.upper()
insert_start = plasmid.find("N" * 80)
right_flank = RafiDataset.RIGHT_FLANK
left_flank = plasmid[insert_start - length : insert_start]

print(RafiDataset.TYPES)

['all', 'high', 'low', 'yeast', 'random', 'challenging', 'snv', 'perturbation', 'tiling']


In [5]:
# preprocessing
train_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0.5),t.Seq2Tensor(),])
val_test_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(0),t.Seq2Tensor(),])

# load the data
train_dataset = RafiDataset(split="train", transform=train_transform, root="../data")
val_dataset = RafiDataset(split="val", data_type=["all"], transform=val_test_transform, root="../data")
test_dataset = RafiDataset(split="test", data_type=["all"], transform=val_test_transform, root="../data")

print(len(train_dataset), len(val_dataset), len(test_dataset))

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

in_channels = len(train_dataset[0][0])
out_channels = 1

6739258 9045 62058


In [6]:
model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[1, 1, 1, 1],
    resize_factor=4,
)
model.apply(initialize_weights)

seq_model = LitModel_Rafi(model=model, loss=nn.MSELoss(), weight_decay=1e-2, lr=1e-2, print_each=1)

# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[1],
    max_epochs=1,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=False,
    num_sanity_val_steps=0,
    enable_model_summary=False
)
# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(seq_model, dataloaders=test_loader)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Loading `train_dataloader` to estimate number of stepping batches.
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(



---------------------------------------------------------------------------------
| Epoch: 0 | Val Loss: 137.23518 | Val Pearson: 0.93692 | Train Pearson: 0.51794 
---------------------------------------------------------------------------------



`Trainer.fit` stopped: `max_epochs=1` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           135.43704223632812
      test_pearson          0.9261932373046875
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 135.43704223632812, 'test_pearson': 0.9261932373046875}]

## Evaluating Single sequences

In [7]:
forw_transform = t.Compose([t.AddFlanks(left_flank, right_flank), t.LeftCrop(length, length), t.Seq2Tensor()])
rev_transform = t.Compose([t.AddFlanks(left_flank, right_flank),t.LeftCrop(length, length),t.ReverseComplement(1),t.Seq2Tensor(),])

def meaned_prediction(forw, rev, trainer, seq_model, name, is_paired=False):
    predictions_forw = trainer.predict(seq_model, dataloaders=forw)
    targets = torch.cat([pred["target"] for pred in predictions_forw])
    y_preds_forw = torch.cat([pred["ref_predicted"] for pred in predictions_forw])

    predictions_rev = trainer.predict(seq_model, dataloaders=rev)
    y_preds_rev = torch.cat([pred["ref_predicted"] for pred in predictions_rev])

    mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

    pears = PearsonCorrCoef()
    print("Task '" + name + "' Pearson r^2")

    if is_paired:
        y_preds_forw_alt = torch.cat(
            [pred["alt_predicted"] for pred in predictions_forw]
        )
        y_preds_rev_alt = torch.cat([pred["alt_predicted"] for pred in predictions_rev])
        mean_alt = torch.mean(torch.stack([y_preds_forw_alt, y_preds_rev_alt]), dim=0)
        pred = mean_alt - mean_forw
        return pears(pred, targets) * pears(pred, targets)

    return pears(mean_forw, targets) * pears(mean_forw, targets)

### All Sequences

In [8]:
test_forw = RafiDataset(split="test", data_type=["all"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["all"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

meaned_prediction(forw, rev, trainer, seq_model, "All Sequences")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Task 'All Sequences' Pearson r^2


tensor(0.8630)

### High

In [9]:
test_forw = RafiDataset(split="test", data_type=["high"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["high"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "High")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Task 'High' Pearson r^2


tensor(0.0119)

### Low

In [10]:
test_forw = RafiDataset(split="test", data_type=["low"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["low"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Low")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Task 'Low' Pearson r^2


tensor(0.2481)

### Native

In [11]:
test_forw = RafiDataset(split="test", data_type=["yeast"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["yeast"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Native")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Task 'Native' Pearson r^2


tensor(0.6196)

### Random

In [12]:
test_forw = RafiDataset(split="test", data_type=["random"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["random"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Random")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Task 'Random' Pearson r^2


tensor(0.9230)

### Challenging

In [13]:
test_forw = RafiDataset(split="test", data_type=["challenging"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["challenging"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Challenging")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Task 'Challenging' Pearson r^2


tensor(0.7398)

## Evaluating Paired sequences

### SNVs

In [14]:
test_forw = RafiDataset(split="test", data_type=["snv"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["snv"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "SNVs", is_paired=True)

100%|██████████| 14.6M/14.6M [00:03<00:00, 4.28MB/s]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Task 'SNVs' Pearson r^2


tensor(0.6026)

### Motif Perturbation

In [15]:
test_forw = RafiDataset(split="test", data_type=["perturbation"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["perturbation"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Motif Perturbation", is_paired=True)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Task 'Motif Perturbation' Pearson r^2


tensor(0.8934)

### Motif Tiling

In [16]:
test_forw = RafiDataset(split="test", data_type=["tiling"], transform=forw_transform, root="../data")
test_rev = RafiDataset(split="test", data_type=["tiling"], transform=rev_transform, root="../data")

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)


meaned_prediction(forw, rev, trainer, seq_model, "Motif Tiling", is_paired=True)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Task 'Motif Tiling' Pearson r^2


tensor(0.7103)